# PPMI 行列の SVD で単語ベクトルを作る（カウントベース）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/B_implementations/ppmi_svd.ipynb)

NLP クックブック B-07 ／ 仕組みを理解するための実装

ニューラルネットを使わずに、共起行列から単語ベクトルを作りたい。word2vec が何をしているかの理解にも役立つ。

In [1]:
import nltk
nltk.download("brown", quiet=True)
from collections import Counter
from nltk.corpus import brown
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

sents = [[w.lower() for w in s if w.isalpha()] for s in brown.sents()]
freq = Counter(w for s in sents for w in s)
vocab = [w for w, c in freq.most_common(8000)]
idx = {w: i for i, w in enumerate(vocab)}

window, cooc = 4, Counter()
for s in sents:
    ids = [idx[w] for w in s if w in idx]
    for i, a in enumerate(ids):
        for b in ids[max(0, i - window):i]:
            cooc[a, b] += 1
            cooc[b, a] += 1
print(len(cooc), "組の共起")

1625258 組の共起


In [2]:
rows, cols = zip(*cooc.keys())
C = csr_matrix((list(cooc.values()), (rows, cols)), shape=(len(vocab), len(vocab)), dtype=np.float64)
total, row_sum = C.sum(), np.asarray(C.sum(axis=1)).ravel()

coo = C.tocoo()
pmi = np.log(coo.data * total / (row_sum[coo.row] * row_sum[coo.col]))
ppmi = csr_matrix((np.maximum(pmi, 0), (coo.row, coo.col)), shape=C.shape)   # 負の値は 0 に

U, S, _ = svds(ppmi, k=100)
vecs = U * np.sqrt(S)
vecs /= np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-9

def most_similar(word, topn=6):
    sims = vecs @ vecs[idx[word]]
    return [vocab[i] for i in np.argsort(-sims)[1:topn + 1]]

for w in ["money", "school", "monday", "walked"]:
    print(w, most_similar(w))

money ['bills', 'pay', 'farming', 'taxes', 'amount', 'cash']
school ['schools', 'graduate', 'college', 'university', 'teachers', 'elementary']
monday ['wednesday', 'tuesday', 'saturday', 'friday', 'sunday', 'thursday']
walked ['parked', 'climbed', 'lobby', 'front', 'sprang', 'rear']


- skip-gram with negative sampling は、（シフトした）PMI 行列の行列分解を暗黙に行っていることが知られている (Levy & Goldberg, 2014)。